# KL Direction Explorer

Chapter 14: Information Theory · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare KL(p||q) and KL(q||p) for two one-dimensional Gaussian distributions.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-14/kl-direction/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
from torch.distributions import Normal, kl_divergence

# The demo's defaults: p = N(0, 1), q = N(mu, s^2)
mu, s = 1.0, 0.7
p = Normal(torch.tensor(0.0, dtype=torch.float64), torch.tensor(1.0, dtype=torch.float64))
q = Normal(torch.tensor(mu, dtype=torch.float64), torch.tensor(s, dtype=torch.float64))


def kl_closed(m1, s1, m2, s2):
    """KL(N(m1, s1^2) || N(m2, s2^2))."""
    return math.log(s2 / s1) + (s1 ** 2 + (m1 - m2) ** 2) / (2 * s2 ** 2) - 0.5


print("KL(p || q) closed form =", kl_closed(0, 1, mu, s), " torch =", kl_divergence(p, q).item())
print("KL(q || p) closed form =", kl_closed(mu, s, 0, 1), " torch =", kl_divergence(q, p).item())
print("-> KL is not symmetric")

# ── Numerical integration on a grid ─────────────────────────────────────────
x = torch.linspace(-10, 10, 40001, dtype=torch.float64)
lp, lq = p.log_prob(x), q.log_prob(x)
print("\ngrid KL(p||q) =", torch.trapezoid(lp.exp() * (lp - lq), x).item())

# ── Monte Carlo: KL(p||q) = E_p[log p(X) - log q(X)] ────────────────────────
torch.manual_seed(0)
xs = p.sample((500000,))
print("Monte Carlo KL(p||q) =", (p.log_prob(xs) - q.log_prob(xs)).mean().item())

# A narrow q pays a lot under KL(p||q) (p puts mass where q is tiny)
print("\n  s_q   KL(p||q)   KL(q||p)")
for sq in (0.3, 0.7, 1.0, 1.5, 3.0):
    print(f"{sq:5.1f}   {kl_closed(0, 1, mu, sq):8.4f}   {kl_closed(mu, sq, 0, 1):8.4f}")

KL(p || q) closed form = 1.1841413825918803  torch = 1.1841413825918798
KL(q || p) closed form = 0.6016749439387323  torch = 0.6016749439387324
-> KL is not symmetric

grid KL(p||q) = 1.18414138259188
Monte Carlo KL(p||q) = 1.1833038917030378

  s_q   KL(p||q)   KL(q||p)
  0.3     9.4071     1.2490
  0.7     1.1841     0.6017
  1.0     0.5000     0.5000
  1.5     0.3499     0.7195
  3.0     0.7097     3.4014
